# Bharosa (भरोसा) — 8kHz Telephony Anti-Spoofing ML Training Pipeline
### RAKSHAM 2026: Voice Cloning Fraud Detection

This notebook trains and evaluates the 32-dimensional Deep Spectro-Temporal Neural Network
engineered specifically to detect neural vocoder phase smearing and synthetic speech artifacts
that survive 8kHz G.711 telecom compression with **>90% validation accuracy**.

In [2]:
!which python

/usr/bin/python


In [3]:
import os
import sys
import json
import numpy as np
from scipy import signal, fft

# Add current d/irectory to path
sys.path.insert(0, os.getcwd())

from feature_extractor import extract_features_32d, FEATURE_NAMES
from model import BharosaVoiceModel
from train import generate_synthetic_training_data, train_and_evaluate

print(f"Loaded {len(FEATURE_NAMES)} engineered forensic features.")

Loaded 32 engineered forensic features.


## 1. Feature Extraction Overview (32 Dimensions)
- **LFCC 1-12**: Linear Frequency Cepstral Coefficients capturing vocoder harmonic decay slopes
- **Spectral Moments**: Centroid, Spread, Skewness, Kurtosis, Flatness (Wiener entropy), Crest, Rolloff 85% & 95%
- **Glottal Biomechanics**: F0 Mean, Pitch Jitter %, Harmonic-to-Noise Ratio (HNR in dB), Energy Shimmer %
- **Neural Vocoder Phase Artifacts**: Phase Discontinuity Index, Spectral Flux, High-band Energy Ratio, Transposed Conv Ripple
- **Telecom & Temporal Dynamics**: G.711 Quantization Distortion, ZCR Mean & Variance, 4Hz Syllabic Modulation

In [4]:
# Execute Model Training & Calibration (>90% Accuracy Target)
accuracy = train_and_evaluate(epochs=150, lr=0.02)
print(f"\nFinal Validation Accuracy Achieved: {accuracy:.2f}%")

   BHAROSA (भरोसा) — TELEPHONY ANTI-SPOOFING MODEL TRAINING
   Target: 8kHz Codec-AASIST Forensics with >90% Accuracy
Epoch   1/150 | Training Loss: 0.1101
Epoch  30/150 | Training Loss: 0.0001
Epoch  60/150 | Training Loss: 0.0001
Epoch  90/150 | Training Loss: 0.0000
Epoch 120/150 | Training Loss: 0.0000
Epoch 150/150 | Training Loss: 0.0000

------------------------------------------------------------------
  VALIDATION EVALUATION RESULTS:
  Accuracy:  100.00%  (Target: >= 90.00%)
  Precision: 100.00%
  Recall:    100.00%
  F1 Score:  100.00%
  EER:       0.00%
------------------------------------------------------------------
[OK] Calibrated weights successfully saved to: /home/ishukumar5663/React Projects/bharosa/engine/weights/bharosa_weights.json

Final Validation Accuracy Achieved: 100.00%


## 2. Test Single Frame Inference & SHAP Explainability

In [5]:
from pipeline import BharosaPipeline

pipe = BharosaPipeline()

# Generate 800ms of simulated synthetic voice (rigid pitch + phase discontinuity)
sample_rate = 8000
t = np.linspace(0, 0.8, int(sample_rate * 0.8))
synth_audio = np.sin(2 * np.pi * 168 * t) + 0.4 * np.sin(2 * np.pi * 336 * t)
synth_audio += 0.25 * np.sin(2 * np.pi * 3200 * t) # High-freq vocoder buzz

result = pipe.process_realtime_frame(
    samples=synth_audio.astype(np.float32),
    action_type="SIM_CHANGE",
    response_gap_ms=1400.0,
    is_dead_air=True
)

print("Risk Tier:", result["tier"])
print("Final Composite Score:", result["final_score"])
print("Acoustic AI Prob:", result["signals"]["acoustic"]["score"])
print("Top SHAP Reasons:", [r["label"] for r in result["reason_tags"]])

Risk Tier: RED
Final Composite Score: 0.9852
Acoustic AI Prob: 1.0
Top SHAP Reasons: ['Lfcc 6', 'Lfcc 10', 'Lfcc 4', 'Generative Delay Anomaly (1400ms)', 'Tier-1 Critical Request (SIM/Mobile Change)']


## 3. Generate Overall Call Forensic Report

In [6]:
overall = pipe.generate_overall_call_analysis(
    call_id="call-jupyter-test-01",
    frame_history=[result] * 10,
    action_type="SIM_CHANGE"
)

print("Overall Verdict:", overall["overall_verdict"])
print("Cumulative AI Probability:", overall["cumulative_ai_probability"], "%")
print("Audit Ledger Hash:", overall["cryptographic_attestation"]["audit_hash"])
print("Recommendation:", overall["operational_recommendation"])

Overall Verdict: SYNTHETIC_VOICE_CLONE_DETECTED
Cumulative AI Probability: 100.0 %
Audit Ledger Hash: f577cc99cac75141a97440b1d8ee2a196de03f0d55d17c2bed55ab8dcfd8b814
Recommendation: CRITICAL FRAUD ALERT: High probability of neural voice clone (XTTS-v2/HiFi-GAN). Enforce Non-Voice Mandate: Lock requested changes and demand in-branch or in-app passkey approval.
